In [1]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from functions import *

In [2]:
model = "Llama-3-8B-Instruct-Gradient-1048k_10_shot"#"Llama-3-8B-Instruct_4_shot"

### Model Output to nice JSON and Failure 

In [3]:
input_dir = f"model_output/{model}/output/"
output_dir = f"model_output/{model}/formatted/"
failure_dir = f"model_output/{model}/failed/"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
if not os.path.exists(failure_dir):
    os.makedirs(failure_dir)

for filename in os.listdir(input_dir):
    if filename.endswith('.json'):
        input_file_path = os.path.join(input_dir, filename)
        output_file_path = os.path.join(output_dir, filename)
        failure_file_path = os.path.join(failure_dir, filename)
        try:
            file = read_json(input_file_path)
            print(f"Processing file: {filename}")
            save_json_to_file(file, output_file_path)
        except Exception as e:
            print(f"Error processing file {filename}: {e}")
            shutil.move(input_file_path, failure_file_path)

Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00050349_exc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00050349_inc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00061308_exc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00061308_inc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00094861_exc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00183885_exc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00183885_inc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00198913_exc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00198913_inc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00235170_exc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00235170_inc_10_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00236340_exc_10_shot.json
Processing file: Llama-3-8B-

In [4]:
p2_label_path = "chia_label/p2"
ready_path = f"model_output/{model}/ready"
failed_model_path = f"model_output/{model}/failed_inner"
p2_model_formatted_path = f"model_output/{model}/formatted"
eval_path = f"evaluate/batch1"

for path in [ready_path, failed_model_path, p2_model_formatted_path]: 
    os.makedirs(path, exist_ok=True)

In [5]:
def extract_nct_number(filename):
    parts = filename.split('_')
    nct_number = None
    file_type = None
    for part in parts:
        if part.startswith("NCT"):
            nct_number = part
        if part in ["inc", "exc"]:
            file_type = part
    return nct_number+"_"+file_type


def extract_logical_structure(data):
    structure = defaultdict(int)
    def traverse(node, depth=0):
        nonlocal structure
        structure["depth"] = max(structure["depth"], depth)

        if isinstance(node, dict):
            for key in node:
                if key in ["AND", "OR", "NOT"]:
                    structure[key] += 1
                traverse(node[key], depth + 1)
        elif isinstance(node, list):
            for item in node:
                traverse(item, depth + 1)

    traverse(data)
    return dict(structure)

def extract_raw_texts(data):
    raw_texts = set()
    def traverse(node):
        if isinstance(node, dict):
            for key, value in node.items():
                if key == "raw_text":
                    raw_texts.add(value)
                else:
                    traverse(value)
        elif isinstance(node, list):
            for item in node:
                traverse(item)

    traverse(data)
    return raw_texts

def extract_words(raw_texts):
    words = set()
    for text in raw_texts:
        words.update(text.split())
    return words

In [6]:
label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}

common_ncts = set(label_files.keys()).intersection(model_files.keys())

In [7]:
labels = []
predictions = []
success_data = []

In [9]:
for nct in common_ncts:# ["NCT00050349_exc"]
    try:
        label_data = read_json(label_files[nct])
        model_data = read_json(model_files[nct])
        label_structure = extract_logical_structure(label_data)
        model_structure = extract_logical_structure(model_data)

        # Try count words
        label_raw_texts = extract_raw_texts(label_data)
        model_raw_texts = extract_raw_texts(model_data)
        missing_texts = label_raw_texts - model_raw_texts
        label_words = extract_words(label_raw_texts)
        model_words = extract_words(model_raw_texts)
        missing_words = label_words - model_words
        missing_words_pct = len(missing_words) / len(label_words) * 100 if label_words else 0
        print(missing_words_pct)
        
        success_data.append({
            'NCT': nct,
            'label_AND': label_structure.get('AND', 0),
            'label_OR': label_structure.get('OR', 0),
            'label_NOT': label_structure.get('NOT', 0),
            'label_DEPTH': label_structure.get('depth', 0),
            'model_AND': model_structure.get('AND', 0),
            'model_OR': model_structure.get('OR', 0),
            'model_NOT': model_structure.get('NOT', 0),
            'model_DEPTH': model_structure.get('depth', 0),
            'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
            'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
            'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
            'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0
            
        })

        labels.append(label_structure)
        predictions.append(model_structure)
        shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
    except Exception as e:
        print(f"Error processing NCT {nct}: {e}")
        shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))

Error processing NCT NCT02041299_inc: Expecting ',' delimiter: line 37 column 2 (char 1291)
Error processing NCT NCT02055053_exc: Expecting ',' delimiter: line 32 column 2 (char 644)
Error processing NCT NCT02773173_inc: Expecting ',' delimiter: line 31 column 2 (char 717)
Error processing NCT NCT02277067_exc: Expecting ',' delimiter: line 17 column 2 (char 421)
0.0
Error processing NCT NCT02653131_inc: Expecting ',' delimiter: line 20 column 2 (char 402)
8.695652173913043
0.0
Error processing NCT NCT02701777_inc: Extra data: line 85 column 1 (char 2530)
Error processing NCT NCT00959569_inc: Expecting ',' delimiter: line 20 column 2 (char 359)
Error processing NCT NCT01088750_inc: Expecting ',' delimiter: line 24 column 2 (char 384)
0.0
0.0
Error processing NCT NCT02607748_inc: Expecting ',' delimiter: line 42 column 2 (char 1793)
3.125
0.0
Error processing NCT NCT02787863_inc: Expecting ',' delimiter: line 49 column 2 (char 2253)
Error processing NCT NCT02926989_exc: Extra data: line 

In [10]:
df_success = pd.DataFrame(success_data).set_index('NCT')
df_success.to_csv(eval_path+f'/{model}_eval.csv')

df_success

,label_AND,label_OR,label_NOT,label_DEPTH,model_AND,model_OR,model_NOT,model_DEPTH,diff_AND,diff_OR,diff_NOT,diff_DEPTH
NCT,,,,,,,,,,,,
NCT02966236_inc,2,0,0,5,1,0,0,3,-1,0,0,-1
NCT02566226_exc,6,0,0,11,1,4,0,11,-1,1,0,0
NCT02933671_inc,3,0,0,7,3,1,0,8,0,1,0,1
NCT02816164_inc,4,6,0,17,5,1,0,13,1,-1,0,-1
NCT01944800_exc,1,0,0,3,1,1,0,5,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
NCT01631058_exc,3,2,0,9,1,2,0,7,-1,0,0,-1
NCT02200978_exc,1,3,0,9,1,1,0,5,0,-1,0,-1
NCT02469610_exc,1,0,0,3,1,0,0,3,0,0,0,0


In [11]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    diffs = y_pred - y_true
    pct_greater = (diffs > 0).sum() / len(diffs) * 100
    pct_less = (diffs < 0).sum() / len(diffs) * 100
    pct_equal = (diffs == 0).sum() / len(diffs) * 100


    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'pct_greater': round(pct_greater, 2),
        'pct_less': round(pct_less, 2),
        'pct_equal': round(pct_equal, 2)
       # 'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    metrics_df = pd.DataFrame(metrics).T  
    num_nct_files = len(df_success)
    metrics_df['num_nct_files'] = num_nct_files
    metrics_df['model_name'] = model
    # Save Metrics to CSV
    metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))

print(f"{len(df_success)} Daten mit {model}")
for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    print(f"  Precision: {values['precision']}")
    print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")

    print(f"  % Greater: {values['pct_greater']}")
    print(f"  % Less: {values['pct_less']}")
    print(f"  % Equal: {values['pct_equal']}")
    print()
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

1032 Daten mit Llama-3-8B-Instruct-Gradient-1048k_10_shot
Metrics for AND:
  Accuracy: 0.254
  Precision: 0.211
  Recall: 0.254
  F1 Score: 0.188
  % Greater: 6.78
  % Less: 67.83
  % Equal: 25.39

Metrics for OR:
  Accuracy: 0.304
  Precision: 0.416
  Recall: 0.304
  F1 Score: 0.342
  % Greater: 48.06
  % Less: 21.51
  % Equal: 30.43

Metrics for NOT:
  Accuracy: 0.859
  Precision: 0.737
  Recall: 0.859
  F1 Score: 0.793
  % Greater: 0.0
  % Less: 14.15
  % Equal: 85.85

Metrics for DEPTH:
  Accuracy: 0.277
  Precision: 0.315
  Recall: 0.277
  F1 Score: 0.284
  % Greater: 32.56
  % Less: 39.73
  % Equal: 27.71



In [12]:
matching_rows = df_success[df_success['label_AND'] == df_success['model_AND']][['label_AND', 'model_AND']]
matching_rows

,label_AND,model_AND
NCT,,
NCT02933671_inc,3,3
NCT01944800_exc,1,1
NCT02566863_inc,2,2
NCT02456532_inc,1,1
NCT01236417_inc,4,4
...,...,...
NCT02323399_exc,1,1
NCT03012984_inc,5,5
NCT01184638_inc,2,2
